In [ ]:
%pip install git+https://github.com/Eickland/HumSpectra.git --upgrade

In [1]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac
from tensorly.decomposition import non_negative_parafac, non_negative_parafac_hals
import matplotlib.pyplot as plt
from scipy import stats
from mpl_toolkits.axes_grid1 import make_axes_locatable

import HumSpectra.fluorescence as fl
from HumSpectra.parafac import EEMDataLoader, OpticalDataAnalyzer, ComponentVisualizer

In [2]:
wdir_2025 = Path.cwd().parent / "initial data" / "2025"
fluo_paths = Path.joinpath(wdir_2025, "Fluorescence")

In [3]:
loader = EEMDataLoader(fluo_paths)
eem_tensor,em_waves,ex_waves,sp_names = loader.load_eem_data()
data_info = loader.get_data_info()

Создан тензор размерности: (108, 70, 40)
Образцы: 108
Длины волн испускания: 70
Длины волн возбуждения: 40


In [4]:
cleaned, mask, residuals, info = loader.robust_detect_outliers(
        eem_tensor, rank=4, threshold_std=3.0, verbose=True
    )

Обнаружено выбросов: 7090


In [5]:
print(f"Процент выбросов: {info['n_outliers'] / eem_tensor.size * 100:.2f}%")
print(f"Обнаружено выбросов: {info['n_outliers']}")

Процент выбросов: 2.34%
Обнаружено выбросов: 7090


In [6]:
analyzer = ComponentVisualizer()
factors = analyzer.fit_parafac(
    cleaned,
    excitation_wavelengths=ex_waves,
    emission_wavelengths=em_waves,
    sample_names=sp_names)


=== ВАЛИДАЦИЯ РЕЗУЛЬТАТОВ ===
Количество факторов: 3
✅ Фактор 0: неотрицательный (min = 1.67e-03)
✅ Фактор 1: неотрицательный (min = 0.00e+00)
✅ Фактор 2: неотрицательный (min = 0.00e+00)
Ошибка реконструкции: 9.85e-02
Сходимость: 9.75e-08
Фактор 0: форма (108, 3), диапазон [0.002, 0.367]
Фактор 1: форма (70, 3), диапазон [0.000, 8.054]
Фактор 2: форма (40, 3), диапазон [0.000, 1.869]

=== МЕТРИКИ КАЧЕСТВА ===
Объясненная дисперсия: 99.03%
Core Consistency: 99.30%


In [8]:
analyzer.export_loadings_to_csv(Path.joinpath(Path.cwd().parent,"Server","Received data","Parafac","parafac.csv"))

Таблица сохранена в c:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Parafac\parafac.csv


In [11]:
pd.read_csv(r'C:\Users\mnbv2\Desktop\lab\KNP analysis\Server\Received data\Parafac\parafac.csv')

,Unnamed: 0,Component_1,Component_2,Component_3
0,ADOM-SL1-1,38.4265,4.0591,57.5144
1,ADOM-SL1-10,35.5910,4.2143,60.1947
2,ADOM-SL1-3,49.1886,9.9172,40.8942
3,ADOM-SL1-4,37.1564,7.0889,55.7547
4,ADOM-SL1-5,26.7243,6.6780,66.5977
...,...,...,...,...
103,ADOM-SLB-B3-1,40.0693,8.1386,51.7921
104,ADOM-SLB-B3-2,38.3113,8.2756,53.4131
105,ADOM-SLB-B3-3,36.6559,6.9741,56.3700
106,ADOM-SLB-BO-2,42.0771,8.9783,48.9447
